# MCDM using Weighted Scoring Method

This notebook applies a weighted scoring method to rank different models based on multiple evaluation metrics. The goal is to select the best model by balancing detection performance (TSS, HSS), ranking ability (PR-AUC), and false positive control (FPR).

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = [
    {
        "Model": "Sliding + Decomposition + Focal",
        "TSS": 0.6980,
        "HSS": 0.3489,
        "PR_AUC": 0.3949,
        "FPR": 0.0441
    },
    {
        "Model": "Decomposition LSTM + Focal",
        "TSS": 0.8068,
        "HSS": 0.2818,
        "PR_AUC": 0.4031,
        "FPR": 0.0751
    },
    {
        "Model": "BiLSTM (BCE)",
        "TSS": 0.8049,
        "HSS": 0.2556,
        "PR_AUC": 0.3882,
        "FPR": 0.0861
    },
    {
        "Model": "LSTM (Focal)",
        "TSS": 0.7712,
        "HSS": 0.2418,
        "PR_AUC": 0.3668,
        "FPR": 0.0883
    },
    {
        "Model": "Sliding Window LSTM (Focal)",
        "TSS": 0.6435,
        "HSS": 0.3324,
        "PR_AUC": 0.3501,
        "FPR": 0.05
    },
    {
        "Model": "XGBoost",
        "TSS": 0.7371,
        "HSS": 0.2889,
        "PR_AUC": 0.3700,
        "FPR": 0.1984
    },
    {
        "Model": "TCN (Focal)",
        "TSS": 0.7024,
        "HSS": 0.2103,
        "PR_AUC": 0.3426,
        "FPR": 0.0954
    },
    {
        "Model": "Soft Voting Ensemble",
        "TSS": 0.7790,
        "HSS": 0.2326,
        "PR_AUC": 0.4106,
        "FPR": 0.0943
    }
]

df = pd.DataFrame(data)
df

,Model,TSS,HSS,PR_AUC,FPR
0,Sliding + Decomposition + Focal,0.6980,0.3489,0.3949,0.0441
1,Decomposition LSTM + Focal,0.8068,0.2818,0.4031,0.0751
2,BiLSTM (BCE),0.8049,0.2556,0.3882,0.0861
3,LSTM (Focal),0.7712,0.2418,0.3668,0.0883
4,Sliding Window LSTM (Focal),0.6435,0.3324,0.3501,0.0500
5,XGBoost,0.7371,0.2889,0.3700,0.1984
6,TCN (Focal),0.7024,0.2103,0.3426,0.0954
7,Soft Voting Ensemble,0.7790,0.2326,0.4106,0.0943


In [3]:
# Normalize positive metrics (higher is better)
for col in ["TSS", "HSS", "PR_AUC"]:
    df[col + "_norm"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# Normalize FPR as inverse metric (lower is better)
df["FPR_norm"] = (df["FPR"].max() - df["FPR"]) / (df["FPR"].max() - df["FPR"].min())

df[["Model", "TSS_norm", "HSS_norm", "PR_AUC_norm", "FPR_norm"]]

,Model,TSS_norm,HSS_norm,PR_AUC_norm,FPR_norm
0,Sliding + Decomposition + Focal,0.333742,1.000000,0.769118,1.000000
1,Decomposition LSTM + Focal,1.000000,0.515873,0.889706,0.799093
2,BiLSTM (BCE),0.988365,0.326840,0.670588,0.727803
3,LSTM (Focal),0.781996,0.227273,0.355882,0.713545
4,Sliding Window LSTM (Focal),0.000000,0.880952,0.110294,0.961763
5,XGBoost,0.573178,0.567100,0.402941,0.000000
6,TCN (Focal),0.360686,0.000000,0.000000,0.667531
7,Soft Voting Ensemble,0.829761,0.160895,1.000000,0.674660


In [4]:
weights = {
    "TSS_norm": 0.4,
    "HSS_norm": 0.3,
    "PR_AUC_norm": 0.2,
    "FPR_norm": 0.1
}

df["MCDM_Score"] = (
    weights["TSS_norm"] * df["TSS_norm"] +
    weights["HSS_norm"] * df["HSS_norm"] +
    weights["PR_AUC_norm"] * df["PR_AUC_norm"] +
    weights["FPR_norm"] * df["FPR_norm"]
)

df = df.sort_values(by="MCDM_Score", ascending=False).reset_index(drop=True)

display(df[["Model", "TSS", "HSS", "PR_AUC", "FPR", "MCDM_Score"]])

best_model = df.loc[0, "Model"]
best_score = df.loc[0, "MCDM_Score"]
print(f"Best model by MCDM: {best_model} (score = {best_score:.4f})")

,Model,TSS,HSS,PR_AUC,FPR,MCDM_Score
0,Decomposition LSTM + Focal,0.8068,0.2818,0.4031,0.0751,0.812612
1,BiLSTM (BCE),0.8049,0.2556,0.3882,0.0861,0.700296
2,Sliding + Decomposition + Focal,0.6980,0.3489,0.3949,0.0441,0.687320
3,Soft Voting Ensemble,0.7790,0.2326,0.4106,0.0943,0.647639
4,LSTM (Focal),0.7712,0.2418,0.3668,0.0883,0.523511
5,XGBoost,0.7371,0.2889,0.3700,0.1984,0.479989
6,Sliding Window LSTM (Focal),0.6435,0.3324,0.3501,0.0500,0.382521
7,TCN (Focal),0.7024,0.2103,0.3426,0.0954,0.211027


Best model by MCDM: Decomposition LSTM + Focal (score = 0.8126)
